# Part 2 – Curated Dataset

## Step 1: Load and Combine Raw Excel Files

The raw data is stored as separate Excel files for different years (2022–2025).

In this step, I load all Excel files from the raw data directory, add a year column based on the filename, and combine them into a single dataframe for further cleaning and harmonization.

Combining the datasets into one dataframe makes it easier to:
- inspect structural differences between years
- identify inconsistencies
- harmonize column names and data types
- perform unified cleaning and transformation

In [53]:
# This makes it so that jupyter wont just output the last line, but everything that gets outputted.
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

from pathlib import Path
from pydantic import BaseModel
from IPython.display import display


In [19]:
# path to raw data
raw_path = Path("../data/raw")

# find all Excel files
excel_files = raw_path.glob("*.xlsx")

In [20]:
# helper functions
def load_excel(file, sheet_name):
    print(f"File: {file}")
    year = int(file.stem[-4:])
    if year == 2022:
        header_row = 0
    elif year == 2025:
        header_row = 6
    else:
        header_row = 5

    try:
        df = pd.read_excel(file, sheet_name=sheet_name, header=header_row)

        df["source_year"] = year
        df["source_file"] = Path(file).name
        df["source_sheet"] = sheet_name

        return df

    except Exception as e:
        print(f"Failed to load {file.name}: {e}")

        return None


def load_all_years(sheet_name):
    dfs = {}

    for file in excel_files:
        year = file.stem[-4:]
        dfs[year] = load_excel(file, sheet_name)

    return dfs


def check_schema(dfs):
    schema_df = pd.DataFrame(
        [
            {
                "source_year": year,
                "shape": df.shape,
                "columns": list(df.columns),
            }
            for year, df in dfs.items()
        ]
    )

    return schema_df


## Step 2: Comparing Table Structures Across Years

To prepare for harmonization, I inspected the structure of Tabell 3 across all years.

The inspection included:
- dataset dimensions
- column names
- structural differences between years

The comparison revealed that the 2022 dataset contains fewer columns than the datasets from 2023–2025.

This indicates that schema harmonization is required before the datasets can be combined into a unified curated dataset.

In [21]:
# Inspect each dataframe
dfs = load_all_years("Tabell 3")

schema_df = check_schema(dfs)
schema_df

File: ../data/raw/resultat-ansokningsomgang-2022.xlsx
File: ../data/raw/resultat-ansokningsomgang-2023.xlsx
File: ../data/raw/resultat-ansokningsomgang-2024.xlsx
File: ../data/raw/resultat-ansokningsomgang-2025.xlsx


,source_year,shape,columns
0,2022,"(1207, 19)","[Utbildningsområde, Utbildningsnamn, Beslut, D..."
1,2023,"(1258, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."
2,2024,"(1272, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."
3,2025,"(1184, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."


### Schema Comparison Summary

The historical Excel files shared several core business concepts, but their structures were not fully consistent across years.

The main differences were:

- different header row positions across years
- slightly different column names
- inconsistent naming conventions
- some variation in available columns

Because of this, I created a harmonized target schema and mapped the source columns into a consistent structure.

## Step 3: Define Target Schema

Based on the schema comparison, a common target schema was defined for the curated dataset.

### Column Mapping Rationale

The source Excel files used slightly different column names across years for the same business concepts.

Examples:

| Original Columns | Harmonized Column |
|---|---|
| Utbildningsanordnare administrativ enhet | utbildningsanordnare |
| Studietakt % | studietakt_procent |
| Beslut | beslut |
| Studieform | studieform |

The harmonization was based on semantic meaning rather than exact text matching.

In addition, normalized English analytical columns such as `decision_normalized` and `study_form_normalized` were added to support downstream API filtering and analytics use cases while preserving the original Swedish source terminology.

In [33]:
TARGET_COLUMNS = [
    "source_year",
    "source_file",
    "source_sheet",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "decision_normalized",
    "kommun",
    "lan",
    "yh_poang",
    "studieform",
    "study_form_normalized",
    "studietakt_procent",
    "utbildningsanordnare",
    "huvudmannatyp",
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "seqf_niva",
    "smalt_yrkesomrade"
]

COLUMN_MAPPING = {
    "utbildningsanordnare_administrativ_enhet": "utbildningsanordnare",
}

## Step 4: Column Standardization & Basic Cleaning

Column names were standardized before harmonization to reduce technical differences between years.

The cleaning rules included:

- converting all column names to lowercase
- removing leading and trailing spaces
- replacing Swedish characters with ASCII equivalents
- replacing spaces and special characters with underscores
- removing parentheses and percentage symbols where needed

This made it easier to compare columns across years and apply a consistent column mapping.

In [34]:
def clean_column_name(col):
    return (
        str(col)
        .strip()
        .lower()
        .replace("å", "a")
        .replace("ä", "a")
        .replace("ö", "o")
        .replace("%", "procent")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )


def clean_string_values(df):
    df = df.copy()

    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].str.strip()

    return df


# clean all years
standardized_dfs = {}

for year, df in dfs.items():
    df = df.copy()

    # clean column names
    df.columns = [clean_column_name(col) for col in df.columns]

    # clean string values
    df = clean_string_values(df)

    standardized_dfs[year] = df


# check result
schema_df = check_schema(standardized_dfs)
schema_df

,source_year,shape,columns
0,2022,"(1207, 19)","[utbildningsomrade, utbildningsnamn, beslut, d..."
1,2023,"(1258, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."
2,2024,"(1272, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."
3,2025,"(1184, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."


## Step 5: Schema Harmonization

After standardizing the column names, the datasets still contained structural differences between years. In particular, the 2022 dataset included fewer columns than the datasets from 2023–2025.

To create a unified curated dataset, the schemas were harmonized into a common target structure.

The harmonization process included:

- aligning all datasets to the predefined target schema
- adding missing columns with null values
- keeping only relevant columns
- ensuring a consistent column order across all years
- adding source metadata columns for traceability

This step ensures that all yearly datasets can be safely combined into a single curated dataset.

In [35]:
# harmonize single dataframe
def harmonize_schema(df, target_columns):

    df = df.copy()

    # rename source columns to target names
    df = df.rename(columns=COLUMN_MAPPING)

    # create normalized English decision column
    DECISION_MAPPING = {
        "Beviljad": "approved",
        "Avslag": "rejected",
        "Återkallad": "withdrawn",
    }

    if "beslut" in df.columns:
        df["decision_normalized"] = (
            df["beslut"]
            .astype("string")
            .str.strip()
            .map(DECISION_MAPPING)
            .fillna("other")
        )

    # create normalized English study form column
    STUDY_FORM_MAPPING = {
        "Distans": "distance",
        "Bunden": "on_site",
    }

    if "studieform" in df.columns:
        df["study_form_normalized"] = (
            df["studieform"]
            .astype("string")
            .str.strip()
            .map(STUDY_FORM_MAPPING)
            .fillna("other")
        )

    # add missing columns
    for col in target_columns:
        if col not in df.columns:
            df[col] = pd.NA

    # keep only target columns
    df = df[target_columns]

    return df


# harmonize all dataframes
harmonized_dfs = {}
for year, df in standardized_dfs.items():
    harmonized_df = harmonize_schema(df=df, target_columns=TARGET_COLUMNS)

    harmonized_dfs[year] = harmonized_df


schema_df = check_schema(harmonized_dfs)
schema_df


,source_year,shape,columns
0,2022,"(1207, 20)","[source_year, source_file, source_sheet, diari..."
1,2023,"(1258, 20)","[source_year, source_file, source_sheet, diari..."
2,2024,"(1272, 20)","[source_year, source_file, source_sheet, diari..."
3,2025,"(1184, 20)","[source_year, source_file, source_sheet, diari..."


## Step 6: Combine Harmonized Datasets

After harmonizing the schemas across all years, the yearly datasets were combined into a single curated dataset.

Because all datasets now shared the same column structure and naming convention, they could be safely merged into one unified table.

The combined dataset represents application-related information from multiple years in a consistent and analysis-ready format.

This curated dataset will later be used for validation, database integration, and API development.

In [50]:
curated_df = pd.concat(harmonized_dfs.values(), ignore_index=True)
curated_df = curated_df.convert_dtypes()

print("======= CURATED DATASET OVERVIEW =======")
print(f"Rows: {curated_df.shape[0]}")
print(f"Columns: {curated_df.shape[1]}")

curated_df.head()

======= CURATED DATASET OVERVIEW =======
Rows: 4921
Columns: 20


,source_year,source_file,source_sheet,diarienummer,utbildningsnamn,utbildningsomrade,beslut,decision_normalized,kommun,lan,yh_poang,studieform,study_form_normalized,studietakt_procent,utbildningsanordnare,huvudmannatyp,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade
0,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/5458,.NET Cloud developer,Data/IT,Avslag,rejected,Stockholm,Stockholm,400,Bunden,on_site,100,IT-Högskolan Stockholm AB,Privat,<NA>,<NA>,<NA>,<NA>
1,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4695,.NET Developer,Data/IT,Avslag,rejected,Flera kommuner,Flera kommuner,400,Bunden,on_site,100,KYH AB,Privat,<NA>,<NA>,<NA>,<NA>
2,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4476,.NET Utvecklare,Data/IT,Beviljad,approved,Gävle,Gävleborg,400,Bunden,on_site,100,Plushögskolan AB - Teknikhögskolan,Privat,<NA>,<NA>,<NA>,<NA>
3,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4708,.NET Utvecklare,Data/IT,Avslag,rejected,Flera kommuner,Flera kommuner,410,Distans,distance,100,Plushögskolan AB - Teknikhögskolan Mitt,Privat,<NA>,<NA>,<NA>,<NA>
4,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/5529,.NET-utvecklare,Data/IT,Avslag,rejected,Kungälv,Västra Götaland,400,Distans,distance,100,Optimum Education AB,Privat,<NA>,<NA>,<NA>,<NA>


## Step 7: Validation and Quality Checks

Basic exploratory validation checks were performed to inspect the curated dataset after harmonization and merging.

The checks focused on:

- dataset structure and data types
- missing values
- duplicate rows
- categorical consistency
- low-cardinality business fields

These exploratory checks were followed by a more formal validation summary table.

Some columns introduced in later years naturally contain missing values for older datasets.

For example, several SUN5-related classification fields were not available in the 2022 source files and therefore appear as missing values after harmonization.

In [59]:
print("--------- DATASET OVERVIEW ---------")
curated_df.info()

print("\n--------- MISSING VALUES ---------")
display(curated_df.isna().sum())

print("\n--------- DUPLICATE ROWS ---------")
print(curated_df.duplicated().sum())

print("\n--------- LOW-CARDINALITY COLUMN INSPECTION ---------")

inspect_columns = [
    "source_year",
    "decision_normalized",
    "studieform",
    "study_form_normalized",
    "huvudmannatyp",
    "seqf_niva",
]

for col in inspect_columns:
    print(f"\n--- {col} ---")

    value_counts_df = curated_df[col].value_counts(dropna=False).reset_index()

    value_counts_df.columns = [col, "count"]

    display(value_counts_df)


--------- DATASET OVERVIEW ---------
<class 'pandas.DataFrame'>
RangeIndex: 4921 entries, 0 to 4920
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   source_year            4921 non-null   Int64  
 1   source_file            4921 non-null   string 
 2   source_sheet           4921 non-null   string 
 3   diarienummer           4921 non-null   string 
 4   utbildningsnamn        4921 non-null   string 
 5   utbildningsomrade      4921 non-null   string 
 6   beslut                 4921 non-null   string 
 7   decision_normalized    4921 non-null   string 
 8   kommun                 4921 non-null   string 
 9   lan                    4921 non-null   string 
 10  yh_poang               4921 non-null   Int64  
 11  studieform             4921 non-null   string 
 12  study_form_normalized  4921 non-null   string 
 13  studietakt_procent     4921 non-null   Int64  
 14  utbildningsanordnare   4921 no

source_year                 0
source_file                 0
source_sheet                0
diarienummer                0
utbildningsnamn             0
utbildningsomrade           0
beslut                      0
decision_normalized         0
kommun                      0
lan                         0
yh_poang                    0
studieform                  0
study_form_normalized       0
studietakt_procent          0
utbildningsanordnare        0
huvudmannatyp               0
sun5_inriktning          1207
sun5_inriktning_namn     1207
seqf_niva                1260
smalt_yrkesomrade        1207
dtype: int64


--------- DUPLICATE ROWS ---------
0

--------- LOW-CARDINALITY COLUMN INSPECTION ---------

--- source_year ---


,source_year,count
0,2024,1272
1,2023,1258
2,2022,1207
3,2025,1184



--- decision_normalized ---


...

,study_form_normalized,count
0,on_site,2572
1,distance,2349



--- huvudmannatyp ---


,huvudmannatyp,count
0,Privat,4073
1,Kommun,779
2,Region,60
3,Statlig,9



--- seqf_niva ---


,seqf_niva,count
0,5.0,3591
1,<NA>,1260
2,6.0,70


## Formal Validation Summary

In addition to exploratory inspection, a structured validation summary was created to identify potential data quality issues in a more standardized way.

Some missing values were expected due to historical schema differences between years.

For example, several SUN5-related classification fields were not available in the 2022 source files and therefore appear as missing values after harmonization.

In [52]:
validation_summary = pd.DataFrame(
    [
        {
            "check": "missing_diarienummer",
            "affected_rows": curated_df["diarienummer"].isna().sum(),
            "severity": "critical",
        },
        {
            "check": "duplicate_diarienummer",
            "affected_rows": curated_df["diarienummer"].duplicated().sum(),
            "severity": "warning",
        },
        {
            "check": "missing_utbildningsanordnare",
            "affected_rows": curated_df["utbildningsanordnare"].isna().sum(),
            "severity": "warning",
        },
        {
            "check": "invalid_decision_values",
            "affected_rows": (
                ~curated_df["decision_normalized"].isin(
                    [
                        "approved",
                        "rejected",
                        "withdrawn",
                        "other",
                    ]
                )
            ).sum(),
            "severity": "warning",
        },
        {
            "check": "invalid_study_form_values",
            "affected_rows": (
                ~curated_df["study_form_normalized"].isin(
                    [
                        "distance",
                        "on_site",
                        "other",
                    ]
                )
            ).sum(),
            "severity": "warning",
        },
        {
            "check": "missing_sun5_fields",
            "affected_rows": curated_df["sun5_inriktning"].isna().sum(),
            "severity": "info",
        }
    ]
)

validation_summary

,check,affected_rows,severity
0,missing_diarienummer,0,critical
1,duplicate_diarienummer,0,warning
2,missing_utbildningsanordnare,0,warning
3,invalid_decision_values,0,warning
4,invalid_study_form_values,0,warning
5,missing_sun5_fields,1207,info


## Step 8: Export Curated Dataset

After validation and quality checks, the curated dataset was exported for further use.

The exported file represents the cleaned, harmonized, and combined dataset. It can later be loaded into a database and used as the data source for an API.

The dataset was exported as a CSV file to make it easy to inspect and reuse in the next part of the assignment.

In [18]:
output_path = Path("../data/curated")
output_path.mkdir(parents= True, exist_ok=True)

curated_df.to_csv(
  output_path/"curated_applications.csv",
  index=False
)


## Reflection

Working with historical Excel files highlighted several common data engineering challenges, including inconsistent schemas, missing columns, and differences in formatting between years.

Defining a target schema before merging the datasets made the harmonization process more structured and easier to manage.

This assignment also demonstrated the importance of cleaning and validating data before loading it into a database or exposing it through an API.